In [280]:
# Biblioteker
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects
from scipy.stats import chi2
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

In [281]:
# Indlæs data
GDP = pd.read_csv("GDP.csv", skiprows=4)
ODR = pd.read_csv("age-dependency-ratio-old.csv")
TFP = pd.read_csv("total-factor-productivity.csv")
HC_DATA = pd.read_excel("pwt110.xlsx", sheet_name="Data")
HC = HC_DATA[['country','year','hc','ctfp','rgdpe','pop']] # Relevante kolonner fra filen HC_DATA (pwt110)

Formålet er at konstruere et tværsnitsdatasæt med ét datapunkt pr. land, hvor vi forklarer TFP-vækst fra 2002 til 2020 med initial ODR og kontroller fra 2002

In [282]:
# Udvælger relevante variabler fra HC_DATA
HC = HC_DATA[['country','year','hc','ctfp','rgdpe','pop']].copy()

# Beregner BNP per capita ud fra rgdpe og befolkning
HC['gdp_pc'] = HC['rgdpe'] / HC['pop']

# Udtrækker observationsåret 2002 og beholder de variable, der skal bruges som initiale forklarende variable i tværsnittet
base2002 = HC[HC['year'] == 2002][['country','hc','ctfp','gdp_pc']].copy()

# Omdøber variablerne, så det tydeligt fremgår, at de er målt i 2002
base2002.columns = ['country','HC2002','TFP2002','GDPpc2002']

# Udtrækker TFP i slutåret 2020
tfp2020 = HC[HC['year'] == 2020][['country','ctfp']].copy()

# Omdøber TFP-variablen for 2020
tfp2020.columns = ['country','TFP2020']

# Sammenfletter datasæt

Datasættene er filtreret og indeholder kun de relevante kolonner. 

Benytter følgende formel:

Δ log(TFPi​) = log(TFP2020​) − log(TFP2002​)

som viser ændringen i log TFP fra periodens start (fx 2002) til slut (fx 2020)

In [283]:
# Merger 2002- og 2020-data, så hvert land får både initial og slut-TFP
Tvaersnit = pd.merge(base2002, tfp2020, on='country', how='inner')

# Konstruerer long-difference outcome som ændringen i log(TFP) fra 2002 til 2020
Tvaersnit['GrowthTFP'] = np.log(Tvaersnit['TFP2020']) - np.log(Tvaersnit['TFP2002'])

# Udtrækker old dependency ratio i startåret 2002
odr2002 = ODR[ODR['Year'] == 2002][[
    'Entity',
    'Age dependency ratio, old (% of working-age population)'
]].copy()

# Omdøber variablerne, så de matcher merge-strukturen
odr2002.columns = ['country', 'ODR2002']

# Sammenfletter ODR med tværsnittet, så hvert land får sin initiale ODR-værdi
Tvaersnit = pd.merge(Tvaersnit, odr2002, on='country', how='inner')

# Fjerner observationer med manglende værdier
Tvaersnit = Tvaersnit.dropna()

In [284]:
print(Tvaersnit.describe())

           HC2002     TFP2002     GDPpc2002     TFP2020   GrowthTFP  \
count  108.000000  108.000000    108.000000  108.000000  108.000000   
mean     2.500757    0.674189  18667.498333    0.637384   -0.014267   
std      0.659964    0.296350  17809.178627    0.220858    0.304781   
min      1.088122    0.167584    728.561351    0.200132   -1.073842   
25%      2.082968    0.403153   4843.455089    0.471157   -0.224303   
50%      2.576247    0.652114  11166.490440    0.625006   -0.045691   
75%      2.967072    0.883181  32387.904671    0.819311    0.183162   
max      3.598119    1.394810  84592.225755    1.236640    0.942964   

          ODR2002  
count  108.000000  
mean    12.640748  
std      7.701428  
min      1.583193  
25%      6.062487  
50%      8.882906  
75%     20.099692  
max     28.059470  


### Model 1: Growth_TFP = α + β ODR2002 + ε

Har lande med højere ODR i 2002 lavere TFP-vækst 2002–2020?

In [285]:
model1 = smf.ols("GrowthTFP ~ ODR2002", data=Tvaersnit).fit(cov_type='HC1')
print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:              GrowthTFP   R-squared:                       0.069
Model:                            OLS   Adj. R-squared:                  0.060
Method:                 Least Squares   F-statistic:                     11.21
Date:                Wed, 22 Apr 2026   Prob (F-statistic):            0.00113
Time:                        10:26:21   Log-Likelihood:                -20.581
No. Observations:                 108   AIC:                             45.16
Df Residuals:                     106   BIC:                             50.53
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.1168      0.056      2.091      0.0

### Konklussion 

Baseline-regressionen (Model 1) viser en negativ sammenhæng mellem initial old dependency ratio (ODR2002) og efterfølgende vækst i total factor productivity (GrowthTFP). Koefficienten på ODR2002 er -0.0104 og statistisk signifikant (p = 0.006), hvilket indikerer, at lande med relativt flere ældre i forhold til befolkningen i arbejdsalderen i 2002 oplevede lavere TFP-vækst i perioden 2002–2020.

### Model 2: Growth_TFP = α + βODR2002 + γ HC2002 + ε

Påvirker aldring stadig TFP-vækst, når vi tager højde for human capital?

In [286]:
model2 = smf.ols("GrowthTFP ~ ODR2002 + HC2002", data=Tvaersnit).fit(cov_type='HC1')
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:              GrowthTFP   R-squared:                       0.074
Model:                            OLS   Adj. R-squared:                  0.056
Method:                 Least Squares   F-statistic:                     6.689
Date:                Wed, 22 Apr 2026   Prob (F-statistic):            0.00184
Time:                        10:26:21   Log-Likelihood:                -20.274
No. Observations:                 108   AIC:                             46.55
Df Residuals:                     105   BIC:                             54.60
Df Model:                           2                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0321      0.141      0.228      0.8

### Konklusion 

Model 2 styrker Model 1, da koefficienten på ODR2002 fortsat er negativ og statistisk tydelig, selv efter kontrol for human capital. Dette indikerer, at højere ODR i 2002 stadig hænger sammen med lavere vækst i TFP i perioden 2002–2020.

### Model 3: GrowthTFP = α + β ODR200 2 γ HC2002 + δ log(GDPpc2002) + ε

Påvirker aldring stadig produktivitetsvækst, når vi sammenligner lande med samme humankapital og samme udviklingsniveau ved startåret?

Er aldring stadig vigtig, når to lande starter lige rige?

In [287]:
model3 = smf.ols("GrowthTFP ~ ODR2002 + HC2002 + np.log(GDPpc2002)", data=Tvaersnit).fit(cov_type='HC1')
print(model3.summary())

                            OLS Regression Results                            
Dep. Variable:              GrowthTFP   R-squared:                       0.177
Model:                            OLS   Adj. R-squared:                  0.153
Method:                 Least Squares   F-statistic:                     7.555
Date:                Wed, 22 Apr 2026   Prob (F-statistic):           0.000127
Time:                        10:26:21   Log-Likelihood:                -13.931
No. Observations:                 108   AIC:                             35.86
Df Residuals:                     104   BIC:                             46.59
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             0.8431      0.27

### Konklusion

Model 3 undersøger, om der stadig er en sammenhæng mellem ODR (aldrende befolkning) og vækst i TFP, når der kontrolleres for human capital og initialt BNP per capita. Koefficienten på ODR2002 er fortsat negativ, hvilket er foreneligt med Model 1, men p-værdien stiger, og sammenhængen er derfor ikke længere statistisk tydelig. Dette tyder på, at en del af sammenhængen mellem ODR og TFP-vækst kan hænge sammen med forskelle i udviklingsniveau. Forklaringsgraden stiger samtidig, hvilket viser, at modellen forklarer mere af variationen i TFP-vækst end Model 1 og Model 2.

### Model 4: Δlog(TFP​) = α + β ODR_i,2002​ + γ HC_i,2002 ​+ δ log(TFP_i,2002) ​+ ε_i​

Hænger initial aldring sammen med efterfølgende TFP-vækst, når vi også tager højde for initial humankapital og initialt produktivitetsniveau?

In [288]:
model4 = smf.ols("GrowthTFP ~ ODR2002 + HC2002 + np.log(TFP2002)", data=Tvaersnit).fit(cov_type='HC1')
print(model4.summary())

                            OLS Regression Results                            
Dep. Variable:              GrowthTFP   R-squared:                       0.439
Model:                            OLS   Adj. R-squared:                  0.423
Method:                 Least Squares   F-statistic:                     17.40
Date:                Wed, 22 Apr 2026   Prob (F-statistic):           3.14e-09
Time:                        10:26:21   Log-Likelihood:                 6.8033
No. Observations:                 108   AIC:                            -5.607
Df Residuals:                     104   BIC:                             5.122
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -0.5568      0.152     

### Konklusion

Model 4 viser, at sammenhængen mellem ODR og TFP-vækst forsvinder, når der kontrolleres for human capital og initial produktivitet. Det tyder på, at aldring alene ikke er hovedforklaringen på lavere produktivitetsvækst. I stedet spiller landets startniveau og arbejdsstyrkens kompetencer en større rolle. Modellen forklarer samtidig mere af variationen i TFP-vækst end de tidligere modeller.

### Model 5: Δlog(TFP​) = α + β ODR_i,2002​ + γ HC_i,2002 ​+ δ log(TFP_i,2002) ​+ ε_i​

Model 5 tager højde for outliers / ekstreme observationer.

In [289]:
low = Tvaersnit['GrowthTFP'].quantile(0.01)
high = Tvaersnit['GrowthTFP'].quantile(0.99)

df_trim = Tvaersnit[
    (Tvaersnit['GrowthTFP'] >= low) &
    (Tvaersnit['GrowthTFP'] <= high)
].copy()

model5 = smf.ols(
    "GrowthTFP ~ ODR2002 + HC2002 + np.log(TFP2002)",data=df_trim).fit(cov_type='HC1')

print(model5.summary())

                            OLS Regression Results                            
Dep. Variable:              GrowthTFP   R-squared:                       0.418
Model:                            OLS   Adj. R-squared:                  0.401
Method:                 Least Squares   F-statistic:                     18.71
Date:                Wed, 22 Apr 2026   Prob (F-statistic):           1.04e-09
Time:                        10:26:21   Log-Likelihood:                 21.393
No. Observations:                 104   AIC:                            -34.79
Df Residuals:                     100   BIC:                            -24.21
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -0.4456      0.138     

### Konklusion

Model 5 gentager hovedmodellen efter fjernelse af ekstreme observationer for at teste robustheden af resultaterne. Koefficienten på ODR2002 er fortsat tæt på nul og ikke statistisk signifikant, hvilket viser, at sammenhængen mellem aldring og TFP-vækst fortsat ikke er tydelig. Samtidig er koefficienten på log(TFP2002) fortsat negativ og stærkt signifikant, hvilket understøtter, at landenes initiale produktivitetsniveau har stor betydning for den efterfølgende vækst. Resultaterne tyder derfor på, at hovedkonklusionerne ikke drives af enkelte outliers, men er stabile.



# Regionale dummies som robusthedstest

In [290]:
df_region = Tvaersnit.copy()

# Regioner
europe = [
    'Denmark','Sweden','Norway','Finland','Germany','France','Italy',
    'Spain','Portugal','Netherlands','Belgium','Austria','Switzerland',
    'United Kingdom','Ireland','Poland','Czech Republic','Hungary',
    'Slovakia','Greece'
]

asia = [
    'Japan','China','India','Korea','Thailand','Indonesia','Malaysia',
    'Philippines','Vietnam','Pakistan','Bangladesh','Singapore'
]

africa = [
    'South Africa','Nigeria','Kenya','Egypt','Morocco','Tunisia',
    'Ghana','Ethiopia','Algeria','Angola'
]

latin_america = [
    'Argentina','Brazil','Chile','Colombia','Mexico','Peru',
    'Uruguay','Paraguay','Bolivia','Ecuador'
]

north_america = [
    'United States','Canada'
]

oceania = [
    'Australia','New Zealand'
]

middle_east = [
    'Saudi Arabia','United Arab Emirates','Israel','Jordan',
    'Turkey','Iran'
]

# Dummies
df_region['Europe'] = df_region['country'].isin(europe).astype(int)
df_region['Asia'] = df_region['country'].isin(asia).astype(int)
df_region['Africa'] = df_region['country'].isin(africa).astype(int)
df_region['LatinAmerica'] = df_region['country'].isin(latin_america).astype(int)
df_region['NorthAmerica'] = df_region['country'].isin(north_america).astype(int)
df_region['Oceania'] = df_region['country'].isin(oceania).astype(int)
df_region['MiddleEast'] = df_region['country'].isin(middle_east).astype(int)

# Log TFP
df_region['log_TFP2002'] = np.log(df_region['TFP2002'])

# Regression
model_region = smf.ols(
    "GrowthTFP ~ ODR2002 + HC2002 + log_TFP2002 + Europe + Asia + Africa + LatinAmerica + NorthAmerica + Oceania + MiddleEast",
    data=df_region
).fit(cov_type='HC1')

print(model_region.summary())

                            OLS Regression Results                            
Dep. Variable:              GrowthTFP   R-squared:                       0.502
Model:                            OLS   Adj. R-squared:                  0.451
Method:                 Least Squares   F-statistic:                     14.28
Date:                Wed, 22 Apr 2026   Prob (F-statistic):           3.22e-15
Time:                        10:26:21   Log-Likelihood:                 13.215
No. Observations:                 108   AIC:                            -4.430
Df Residuals:                      97   BIC:                             25.07
Df Model:                          10                                         
Covariance Type:                  HC1                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -0.6169      0.162     -3.798   

In [291]:
# Vælg de variable, der bruges i modellerne
df_model = Tvaersnit[['country',
                      'GrowthTFP',
                      'ODR2002',
                      'HC2002',
                      'GDPpc2002',
                      'TFP2002']].dropna().copy()

# Log-variabler
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])
df_model['log_TFP2002'] = np.log(df_model['TFP2002'])

# Regioner
# EUROPA
europe = [
    'Albania','Austria','Belgium','Bulgaria','Croatia','Cyprus',
    'Czechia','Denmark','Estonia','Finland','France','Germany',
    'Greece','Hungary','Iceland','Ireland','Italy','Latvia',
    'Lithuania','Luxembourg','Malta','Netherlands','Norway',
    'Poland','Portugal','Romania','Serbia','Slovakia','Slovenia',
    'Spain','Sweden','Switzerland','Ukraine','United Kingdom',
    'Armenia'
]

# ASIEN
asia = [
    'Bahrain','China','India','Indonesia','Iraq','Israel','Japan',
    'Jordan','Kazakhstan','Kuwait','Kyrgyzstan','Malaysia',
    'Mongolia','Philippines','Qatar','Saudi Arabia','Singapore',
    'Sri Lanka','Taiwan','Tajikistan','Thailand'
]

# AFRIKA
africa = [
    'Angola','Benin','Botswana','Burkina Faso','Burundi',
    'Cameroon','Central African Republic','Egypt','Eswatini',
    'Gabon','Kenya','Lesotho','Mauritania','Mauritius',
    'Morocco','Mozambique','Namibia','Niger','Nigeria',
    'Rwanda','Senegal','Sierra Leone','South Africa',
    'Sudan','Togo','Tunisia','Zambia','Zimbabwe'
]

# LATINAMERIKA / CARIBIEN
latin_america = [
    'Argentina','Barbados','Brazil','Chile','Colombia',
    'Costa Rica','Dominican Republic','Ecuador','El Salvador',
    'Guatemala','Honduras','Jamaica','Mexico','Nicaragua',
    'Panama','Paraguay','Peru','Trinidad and Tobago','Uruguay'
]

# NORDAMERIKA
north_america = [
    'Canada','United States'
]

# OCEANIEN
oceania = [
    'Australia','Fiji','New Zealand'
]

# Regionale dummies
df_model['Europe'] = df_model['country'].isin(europe).astype(int)
df_model['Asia'] = df_model['country'].isin(asia).astype(int)
df_model['Africa'] = df_model['country'].isin(africa).astype(int)
df_model['LatinAmerica'] = df_model['country'].isin(latin_america).astype(int)
df_model['NorthAmerica'] = df_model['country'].isin(north_america).astype(int)
df_model['Oceania'] = df_model['country'].isin(oceania).astype(int)
df_model['MiddleEast'] = df_model['country'].isin(middle_east).astype(int)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model 1: Kun ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model 2: + HC
X2 = sm.add_constant(df_model[['ODR2002', 'HC2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model 3: + log(GDPpc2002)
X3 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model 4: + log(TFP2002) i stedet for GDP
X4 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_TFP2002']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model 5: Samme som model 4, men med trimmed sample (outliers fjernet)
low = df_model['GrowthTFP'].quantile(0.01)
high = df_model['GrowthTFP'].quantile(0.99)

df_trim = df_model[
    (df_model['GrowthTFP'] >= low) &
    (df_model['GrowthTFP'] <= high)
].copy()

y_trim = df_trim['GrowthTFP']
X5 = sm.add_constant(df_trim[['ODR2002', 'HC2002', 'log_TFP2002']])
model5 = sm.OLS(y_trim, X5).fit(cov_type='HC1')

# Model 6: Regionale dummies
X6 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_TFP2002',
                               'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast']])
model6 = sm.OLS(y, X6).fit(cov_type='HC1')

# Samlet tabel
results = summary_col(
    [model1, model2, model3, model4, model5, model6],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)', '(6)'],
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)


                  (1)        (2)        (3)        (4)        (5)        (6)    
--------------------------------------------------------------------------------
const          0.1168**   0.0321     0.8431***  -0.5568*** -0.4456*** -0.4294** 
               (0.0559)   (0.1409)   (0.2727)   (0.1523)   (0.1384)   (0.1769)  
ODR2002        -0.0104*** -0.0135*** -0.0086*   -0.0010    -0.0013    -0.0013   
               (0.0031)   (0.0048)   (0.0045)   (0.0040)   (0.0037)   (0.0065)  
HC2002                    0.0498     0.1819**   0.1278**   0.1005*    0.1134    
                          (0.0687)   (0.0825)   (0.0616)   (0.0562)   (0.0780)  
log_GDPpc2002                        -0.1298***                                 
                                     (0.0375)                                   
log_TFP2002                                     -0.4680*** -0.3982*** -0.4673***
                                                (0.0709)   (0.0621)   (0.0702)  
Europe                     